In [1]:
import seaborn as sns
import wandb
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.image as mpimg
from matplotlib.ticker import ScalarFormatter

sns.set()


api = wandb.Api()
task_name = "sapg_anymal"
project = api.runs("naoki-shitanda/"+task_name)
metric = "rewards/step"
metric_name = "Episode Rewards"
#target_step = 20 * 10**9
target_steps = [1.25e9, 2.5e9, 3.75e9, 5e9]
#target_steps = [2.5e9, 5e9, 7.5e9, 10e9]

print(len(project))    
task_runs = {
    "CPO2(wo/AdR)": [],
    #"CPO(wo/AdR)": [],
    "SAPG": [],
    "PPO": [],
    #"PBT": [],
}

# 各手法名がrun.tagに含まれているrunを抽出
for key in task_runs.keys():
    for run in project:
        if any(key in tag for tag in run.tags):
            task_runs[key].append(run)


print(task_runs)
for key in task_runs.keys():
    print(key,":", len(task_runs[key]))
    

29
{'CPO2(wo/AdR)': [<Run naoki-shitanda/sapg_anymal/uid_00_0725-1-0.001-1-0.2-0-0-sa-5class-ent0002_24576envs_mixed_expl_learn_param_lf_1p_31_07_02h56m47s_seed2 (failed)>, <Run naoki-shitanda/sapg_anymal/uid_00_0725-1-0.001-1-0.2-0-0-sa-5class-ent0002_24576envs_mixed_expl_learn_param_lf_1p_26_07_02h57m36s_seed4 (failed)>, <Run naoki-shitanda/sapg_anymal/uid_00_0725-1-0.001-1-0.2-0-0-sa-5class-ent0002_24576envs_mixed_expl_learn_param_lf_1p_26_07_02h57m34s_seed3 (failed)>, <Run naoki-shitanda/sapg_anymal/uid_00_0725-1-0.001-1-0.2-0-0-sa-5class-ent0002_24576envs_mixed_expl_learn_param_lf_1p_26_07_00h50m52s_seed1 (crashed)>, <Run naoki-shitanda/sapg_anymal/uid_00_0725-1-0.001-1-0.2-0-0-sa-5class-ent0002_24576envs_mixed_expl_learn_param_lf_1p_26_07_00h49m09s_seed0 (crashed)>], 'SAPG': [<Run naoki-shitanda/sapg_anymal/uid_00_0725-SAPG-24576_ent0002_24576envs_mixed_expl_learn_param_lf_1p_26_07_00h39m33s_seed3 (crashed)>, <Run naoki-shitanda/sapg_anymal/uid_00_0725-SAPG-24576_ent0002_24576env

In [2]:
print(task_runs["SAPG"][0].summary)

{'_runtime': 56317, '_step': 36375, '_timestamp': 1753514361.0120049, 'auxiliary_stats/off_on_grad_similarity': 0, 'auxiliary_stats/off_on_relative_grad_norms': 0, 'auxiliary_stats/off_policy_contrib': {'_type': 'histogram'}, 'auxiliary_stats/on_policy_contrib': {'_type': 'histogram'}, 'episode_lengths/iter': 2427.39990234375, 'episode_lengths/step': 2427.39990234375, 'episode_lengths/time': 2427.39990234375, 'global_step': 14303232000, 'info/e_clip': 0.20000000298023224, 'info/epochs': 36376, 'info/kl': 0.004405077081173658, 'info/last_lr': 1.708593663352076e-05, 'info/lr_mul': 1, 'intr_rewards/entropy_block_0': -18.96848487854004, 'intr_rewards/entropy_block_1': -19.8873348236084, 'intr_rewards/entropy_block_2': -20.80568504333496, 'intr_rewards/entropy_block_3': -21.75986671447754, 'intr_rewards/entropy_block_4': -22.733245849609375, 'intr_rewards/entropy_block_5': -20.596954345703125, 'losses/a_loss': 0.004465200938284397, 'losses/bounds_loss': 0.1985725164413452, 'losses/c_loss': 

In [3]:
import torch
import numpy as np
import scipy.stats as stats
from scipy.stats import ttest_ind

# =========================
# データ抽出＋補間
# =========================

def extract_scores_at_step_interpolated(all_runs_dict, target_step, metric="rewards/step"):
    score_rows = []
    method_names = []
    all_seed_labels = []

    for method_name, runs in all_runs_dict.items():
        seed_scores = []
        seed_labels = []

        seeds = [run.config.get("seed", "not found") for run in runs]
        seed_set = set(seeds)

        for seed in seed_set:
            seed_runs = [run for run in runs if run.config.get("seed") == seed]
            seed_runs = sorted(seed_runs, key=lambda run: run.created_at)
            dfs = []

            for i, run in enumerate(seed_runs):
                df = run.history(keys=[metric, "global_step"])
                df = df.set_index("global_step")
                df = df.dropna()
                if i > 0:
                    df = df.iloc[40:]
                dfs.append(df)

            if len(dfs) == 0:
                continue

            seed_df = pd.concat(dfs, axis=0)
            seed_df = seed_df.rename(columns={metric: f"seed{seed}"})
            if "_step" in seed_df.columns:
                seed_df = seed_df.drop("_step", axis=1)

            seed_df = seed_df.sort_index()
            seed_df = seed_df.interpolate(method="linear", limit_direction="both")

            try:
                value = seed_df.loc[target_step].values[0]
            except KeyError:
                all_steps = seed_df.index.to_numpy()
                if target_step < all_steps[0] or target_step > all_steps[-1]:
                    continue
                lower_idx = max(i for i in range(len(all_steps)) if all_steps[i] <= target_step)
                upper_idx = min(i for i in range(len(all_steps)) if all_steps[i] >= target_step)
                lower = all_steps[lower_idx]
                upper = all_steps[upper_idx]
                val_lower = seed_df.loc[lower].values[0]
                val_upper = seed_df.loc[upper].values[0]
                weight = (target_step - lower) / (upper - lower + 1e-8)
                value = (1 - weight) * val_lower + weight * val_upper

            seed_scores.append(value)
            seed_labels.append(seed)

        if seed_scores:
            score_rows.append(seed_scores)
            method_names.append(method_name)
            all_seed_labels.append(seed_labels)

    max_seeds = max(len(row) for row in score_rows)
    padded = [row + [float('nan')] * (max_seeds - len(row)) for row in score_rows]
    tensor = torch.tensor(padded, dtype=torch.float32)
    return tensor, method_names, all_seed_labels

# =========================
# 信頼区間の計算
# =========================

def confidence_interval(data, confidence=0.95):
    n = len(data)
    mean = data.mean().item()
    std = data.std(unbiased=True).item()
    h = stats.t.ppf((1 + confidence) / 2., n - 1) * std / np.sqrt(n)
    return mean, h

# =========================
# メイン出力処理
# =========================

def evaluate_and_report(all_runs_dict, target_step, metric_name, task_name=""):
    tensor, methods, seeds = extract_scores_at_step_interpolated(all_runs_dict, target_step, metric=metric_name)

    with open("output.txt", "w") as f:
        def print_both(text):  # inner function
            print(text)
            f.write(text + "\n")

        print_both("==========================")
        print_both(f"Task: {task_name}")
        print_both(f"Target Env Step: {target_step}")
        print_both(f"Metric: {metric_name}")
        print_both("--------------------------\n")

        ci_list = []
        for i, method in enumerate(methods):
            mean, h = confidence_interval(tensor[i])
            ci_list.append((method, mean, h, tensor[i].numpy()))
            print_both(f"{method}: {tensor[i].numpy()}")
            print_both(f"seeds: {seeds[i]}")
            print_both(f"mean: {mean:.3f}")
            print_both(f"std: {tensor[i].std(unbiased=True).item():.3f}")
            #print_both(f"95% CI: [{mean - h:.3f}, {mean + h:.3f}]\n")
            print_both("--")
        ci_list_sorted = sorted(ci_list, key=lambda x: x[1], reverse=True)
        top_method = ci_list_sorted[0][0]
        top_mean, top_h, top_scores = ci_list_sorted[0][1], ci_list_sorted[0][2], ci_list_sorted[0][3]
        top_ci_lower = top_mean - top_h

        p_values = {}
        for method, mean, h, scores in ci_list:
            if method == top_method:
                continue
            _, p = ttest_ind(top_scores, scores, equal_var=False)
            p_values[method] = p

        top_tied_methods = [top_method] + [m for m in p_values if p_values[m] >= 0.05]
        sig_worse = [m for m in p_values if m not in top_tied_methods]

        print_both("==========================")
        print_both(f"✅ Top method by mean: {top_method}")
        print_both(f"Mean: {top_mean:.3f}, 95% CI: [{top_mean - top_h:.3f}, {top_mean + top_h:.3f}]\n")

        print_both("------ p-values vs top method (Welch's t-test) ------")
        for method, p in p_values.items():
            print_both(f"{method:15s}: p = {p:.4f}")

        print_both("\n------ Summary ------")
        if len(top_tied_methods) == 1:
            print_both(f"✅ Only '{top_method}' is significantly better than all others (p < 0.05)")
        else:
            print_both(f"✅ Statistically tied top methods (p ≥ 0.05 vs top): {', '.join(top_tied_methods)}")

        if sig_worse:
            print_both(f"⚠️  Methods significantly worse than '{top_method}' (p < 0.05): {', '.join(sig_worse)}")
        else:
            print_both(f"✅ No methods are significantly worse than '{top_method}'")

        print_both("\n--------------------------------------------------")
        print_both(f"Top: {top_method}")
        tied_others = [m for m in top_tied_methods if m != top_method]
        if tied_others:
            print_both(f"No Significant Difference: {', '.join(tied_others)}")
        else:
            print_both("No Significant Difference: (none)")

        if sig_worse:
            print_both(f"Significantly Worse: {', '.join(sig_worse)}")
        else:
            print_both("Significantly Worse: (none)")
        print_both("--------------------------------------------------")
        
        


In [4]:
for target_step in target_steps:
    print(f"Evaluating for target step: {target_step}")
    evaluate_and_report(
        all_runs_dict=task_runs,
        target_step=target_step,
        metric_name=metric,
        task_name=task_name,
    )


Evaluating for target step: 1250000000.0


Task: sapg_anymal
Target Env Step: 1250000000.0
Metric: rewards/step
--------------------------

CPO2(wo/AdR): [68.03852  68.92957  67.670006 68.07128  65.956215]
seeds: [0, 1, 2, 3, 4]
mean: 67.733
std: 1.096
--
SAPG: [69.117    68.422356 68.5156   69.61305  67.78012 ]
seeds: [0, 1, 2, 3, 4]
mean: 68.690
std: 0.701
--
PPO: [67.26551 68.72158 68.76725 66.08142 65.37735]
seeds: [0, 1, 2, 3, 4]
mean: 67.243
std: 1.528
--
✅ Top method by mean: SAPG
Mean: 68.690, 95% CI: [67.820, 69.560]

------ p-values vs top method (Welch's t-test) ------
CPO2(wo/AdR)   : p = 0.1453
PPO            : p = 0.1059

------ Summary ------
✅ Statistically tied top methods (p ≥ 0.05 vs top): SAPG, CPO2(wo/AdR), PPO
✅ No methods are significantly worse than 'SAPG'

--------------------------------------------------
Top: SAPG
No Significant Difference: CPO2(wo/AdR), PPO
Significantly Worse: (none)
--------------------------------------------------
Evaluating for target step: 2500000000.0
Task: sapg_anymal
Target 